# screamingface · Quickstart

Compose several AI models into one **fusion**, run it on a benchmark sample, and see whether the
panel beats its strongest member. Connect → pick → compose → run → compare.

This committed copy runs in mock mode — synthetic questions, deterministic local answers — so it
renders reproducibly on GitHub. The scores demonstrate the flow, not provider quality.

**Going live:** replace the setup call with `sf.setup()` and have three things ready:

1. **AI Gateway** running:
   `cd apps/aigateway && uv sync && uv run uvicorn aigateway.main:app --port 9105`.
   `sf.setup()` finds `http://127.0.0.1:9105`; for any other host, pass `gateway="..."` or set
   `SCREAMINGFACE_GATEWAY_URL`.
2. **Providers connected** in the setup panel, with OAuth or an API key. `sf.models.list()` shows
   models from connected providers.
3. **Hugging Face access** for the real GPQA benchmark: `uv sync --extra datasets`, accept the
   terms at `huggingface.co/datasets/Idavidrein/gpqa`, then `huggingface-cli login`.

## 1 · Connect

In [1]:
import screamingface as sf

# REMOVE mode and static_widgets to run live: sf.setup()
session = sf.setup(mode="mock", static_widgets=True)
session

SetupPanel(state='connected', credentials=<never stored>)

## 2 · Pick models

In [2]:
available = sf.models.list()
available

['codex/gpt-5.5', 'gemini-cli/gemini-2.5-pro', 'anthropic/claude-sonnet-4-6']

## 3 · Compose a URL4-backed fusion

In [3]:
fusion = sf.Fusion(
    "frontier-trio",
    models=available[:3],
    reduce="majority_vote",
    judge=available[0],
)
fusion

Role,Model
Judge,codex/gpt-5.5
Member,gemini-cli/gemini-2.5-pro
Member,anthropic/claude-sonnet-4-6


Ask for the shareable URL4 recipe when you need it:

In [4]:
fusion.url4

"(sf-model://codex/gpt-5.5, sf-model://gemini-cli/gemini-2.5-pro, sf-model://anthropic/claude-sonnet-4-6)!'majority_vote';sf_version=1;sf_name=frontier-trio;sf_judge=codex/gpt-5.5"

## 4 · Run

Each member answers every question once through an embedded URL4 node; the vote and the
best-member baseline reuse the same answers, so nothing is asked twice. Live runs validate
providers and models up front and fail fast with `FusionNotReady` before any call.

In [5]:
run = fusion.evaluate("gpqa", first=20, seed=0)
run

Run(benchmark='GPQA-shaped synthetic science fixture', dataset_source='synthetic-gpqa-shaped', mode='mock', models=('codex/gpt-5.5', 'gemini-cli/gemini-2.5-pro', 'anthropic/claude-sonnet-4-6'), url="(sf-model://codex/gpt-5.5, sf-model://gemini-cli/gemini-2.5-pro, sf-model://anthropic/claude-sonnet-4-6)!'majority_vote';sf_version=1;sf_name=frontier-trio;sf_judge=codex/gpt-5.5", sample_size=20, seed=0, score=100.0, baseline=80.0, gain=20.0, cost_usd=0.0, fusion_name='frontier-trio', reduce='majority_vote', judge='codex/gpt-5.5', incomplete=0, profiles=(), pricing_source='estimate:SDK catalog', pricing_as_of='2026-07-16', prompt_tokens=0, completion_tokens=0, total_tokens=0, model_results=(ModelResult(model='codex/gpt-5.5', score=80.0, prompt_tokens=0, completion_tokens=0, total_tokens=0, cost_usd=0.0, failures=0), ModelResult(model='gemini-cli/gemini-2.5-pro', score=80.0, prompt_tokens=0, completion_tokens=0, total_tokens=0, cost_usd=0.0, failures=0), ModelResult(model='anthropic/claude-sonnet-4-6', score=80.0, prompt_tokens=0, completion_tokens=0, total_tokens=0, cost_usd=0.0, failures=0)), failures=())

## 5 · Compare

In [6]:
{
    "mode": run.mode,
    "provenance": run.dataset_source,
    "sample_size": run.sample_size,
    "score": run.score,
    "baseline": run.baseline,
    "gain": run.gain,
    "cost_usd": run.cost_usd,
}

{'mode': 'mock',
 'provenance': 'synthetic-gpqa-shaped',
 'sample_size': 20,
 'score': 100.0,
 'baseline': 80.0,
 'gain': 20.0,
 'cost_usd': 0.0}

> Gain = fusion score − best member, on the same answers. Positive means the panel
beat its strongest member.

**Next:** [`yaml_quickstart.ipynb`](yaml_quickstart.ipynb) keeps the lineup in a file — or share
this fusion by sending `fusion.url4`.